# Challenge de la Sesión 02 — Entrenamiento de modelos generativos: tu propio VAE

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/02-training-generative-models/challenge.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/02-training-generative-models/challenge.ipynb)

> **📋 Reglas del challenge**
> - **Modalidad:** individual. **Tiempo:** 40 minutos en clase; entrega hasta las **23:59 de hoy** (hora de Colombia).
> - **IA permitida** (ChatGPT, Claude, Codex, Copilot…) **con registro obligatorio** al final del notebook. Sin registro, el componente de reflexión y registro de IA vale 0.0.
> - **Micro-sustentaciones:** el profesor puede pedirte que expliques tu notebook en vivo (2–3 minutos, sin IA). Si no puedes explicarlo, la nota máxima del challenge es 3.0.
> - **Rúbrica:** ejecución técnica **40%** · análisis e interpretación **40%** · reflexión y registro de IA **20%**.
> - **Entrega:** ejecuta todo en orden → *File → Download notebook* (Archivo → Descargar notebook) → súbelo a e-Aulas como **`S02_<codigo>.ipynb`**.

**Antes de empezar:** en Kaggle activa *Settings → Accelerator → GPU T4 x2* (recomendado; en CPU también funciona, solo es más lento) e *Internet → On*. Escribe tu código estudiantil en la primera celda y ejecuta todo **en orden**.

**Tu misión.** Vas a entrenar un autoencoder variacional (VAE) con una configuración asignada a partir de tu código
estudiantil (dimensión latente, β, tasa de aprendizaje y 3 prendas de Fashion-MNIST), compararlo con una línea base,
asesorar a una empresa ficticia y depurar un ciclo de entrenamiento escrito por una IA.

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S02"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # los modelos y datos se descargan de Hugging Face
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "Sin conexión a internet: este notebook descarga modelos y datos de Hugging Face.\n"
            "Kaggle: en el panel derecho abre Settings (Session options) y activa Internet. "
            "La opción solo aparece si tu cuenta tiene el teléfono verificado (kaggle.com/settings). "
            "Luego vuelve a ejecutar esta celda. En Colab el internet viene activado."
        ) from None

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---
ensure("datasets")
ensure("matplotlib")

In [ ]:
# --- Tu configuración personal (no editar: depende de tu STUDENT_ID) ---
import itertools
GARMENTS = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
GARMENTS_ES = ["camiseta/top", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "tenis", "bolso", "botín"]

LATENT_DIM = pick([2, 4, 8, 16])
CONFIG = {
    "latent_dim": LATENT_DIM,
    # Si tu dimensión latente es 8, tu β nunca es 1: así tu modelo siempre difiere de la línea base.
    "beta": pick([0.5, 1, 2, 4] if LATENT_DIM != 8 else [0.5, 2, 4]),
    "lr": pick([1e-3, 3e-3]),
    "classes": list(pick(list(itertools.combinations(range(10), 3)))),
}
CONFIG["class_names"] = [GARMENTS[c] for c in CONFIG["classes"]]

# Línea base: latente 8 y β = 1, con tu misma tasa de aprendizaje y tus mismas prendas.
BASELINE = {"latent_dim": 8, "beta": 1, "lr": CONFIG["lr"], "classes": CONFIG["classes"]}

print("🎯 Tu configuración personal")
print(f"  Dimensión latente : {CONFIG['latent_dim']}")
print(f"  β (peso del KL)   : {CONFIG['beta']}")
print(f"  Tasa de aprendizaje: {CONFIG['lr']}")
print("  Prendas           : " + ", ".join(f"{GARMENTS[c]} ({GARMENTS_ES[c]})" for c in CONFIG["classes"]))
print(f"📏 Línea base        : latente {BASELINE['latent_dim']}, β = {BASELINE['beta']}, misma tasa y mismas prendas")

## Preparación: datos, modelo y funciones de apoyo (ejecutar, no editar)

Esta celda carga **solo tus 3 prendas** de Fashion-MNIST (imágenes de 28 × 28 píxeles con valores en [0, 1],
aplanadas a 784 números) y define las mismas piezas del lab:

- `VAE`: codificador y decodificador MLP (256 unidades ocultas) con **Dropout(0.1)** como regularización.
- `vae_loss`: reconstrucción (BCE sumada sobre los 784 píxeles) + β · KL, promediadas por imagen.
- `train_one_epoch` y `evaluate`: el ciclo de entrenamiento y la evaluación en validación.
- `train_vae(...)`: entrena un modelo completo con una semilla fija (`RUN_SEED`), para que las comparaciones sean justas.
- `summarize(...)`: métricas de validación: **reconstrucción**, **KL**, **−ELBO = reconstrucción + KL** (comparable
  entre modelos con distinto β), el **objetivo** entrenado (reconstrucción + β · KL) y las **dimensiones activas**
  (dimensiones latentes cuya media μ varía entre imágenes: varianza > 0.01, criterio de Burda et al., 2016).

Todas las pérdidas se expresan **por imagen** (en nats). Más bajo es mejor para reconstrucción y −ELBO.

In [ ]:
import time
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset

N_PER_CLASS_TRAIN = 200 if FAST_DEV_RUN else 6_000   # 3 prendas × 6 000 = 18 000 imágenes de entrenamiento
N_PER_CLASS_VAL = 100 if FAST_DEV_RUN else 1_000     # 3 prendas × 1 000 = 3 000 imágenes de validación
EPOCHS = 1 if FAST_DEV_RUN else 8
BATCH_SIZE = 128
HIDDEN_DIM = 256
RUN_SEED = 1234   # la misma semilla en todos los entrenamientos: mismos pesos iniciales y mismo orden de lotes

raw = load_dataset("zalando-datasets/fashion_mnist")

def load_classes(split, classes, per_class, seed=RUN_SEED):
    """Imágenes de las clases indicadas → (tensor N×784 en [0, 1], etiquetas)."""
    subset = raw[split].filter(lambda label: label in classes, input_columns="label").shuffle(seed=seed)
    subset = subset.select(range(min(per_class * len(classes), len(subset))))
    pixels = np.stack([np.asarray(img, dtype=np.float32) for img in subset["image"]]) / 255.0
    return torch.from_numpy(pixels).reshape(len(subset), -1), torch.tensor(subset["label"])

x_train, y_train = load_classes("train", CONFIG["classes"], N_PER_CLASS_TRAIN)
x_val, y_val = load_classes("test", CONFIG["classes"], N_PER_CLASS_VAL)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=512)
print(f"Entrenamiento: {tuple(x_train.shape)} · Validación: {tuple(x_val.shape)} · Épocas: {EPOCHS}")


class VAE(nn.Module):
    """VAE totalmente conectado para imágenes de 28 × 28, con Dropout en las capas ocultas."""

    def __init__(self, latent_dim=8, hidden_dim=HIDDEN_DIM, input_dim=784, p_drop=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(p_drop))
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Dropout(p_drop),
                                     nn.Linear(hidden_dim, input_dim))

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


def vae_loss(logits, x, mu, logvar, beta=1.0):
    """Objetivo por imagen: reconstrucción + beta · KL. Devuelve (objetivo, reconstrucción, KL)."""
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta * kl, recon, kl


def train_one_epoch(model, loader, optimizer, beta=1.0):
    """Ciclo de entrenamiento de referencia (el mismo del lab). Devuelve el objetivo promedio por imagen."""
    model.train()
    total, n = 0.0, 0
    for x, _ in loader:
        x = x.to(DEVICE)
        optimizer.zero_grad()
        logits, mu, logvar = model(x)
        loss, recon, kl = vae_loss(logits, x, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n


@torch.no_grad()
def evaluate(model, loader, beta=1.0):
    """Reconstrucción, KL, objetivo y −ELBO promedio por imagen, sin actualizar el modelo."""
    model.eval()
    totals, n = np.zeros(2), 0
    for x, _ in loader:
        x = x.to(DEVICE)
        logits, mu, logvar = model(x)
        _, recon, kl = vae_loss(logits, x, mu, logvar, beta)
        totals += np.array([recon.item(), kl.item()]) * x.size(0)
        n += x.size(0)
    recon, kl = totals / n
    return {"recon": recon, "kl": kl, "objective": recon + beta * kl, "neg_elbo": recon + kl}


def train_vae(latent_dim, beta, lr, epochs=EPOCHS, train_fn=train_one_epoch, label="VAE"):
    """Entrena un VAE desde cero con la semilla RUN_SEED. Devuelve (modelo, historial por época)."""
    torch.manual_seed(RUN_SEED)
    model = VAE(latent_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(RUN_SEED))
    history = {"epoch": [], "train_objective": [], "val_recon": [], "val_kl": [], "val_neg_elbo": []}
    start = time.time()
    print(f"▶ {label}: latente {latent_dim}, β = {beta}, lr = {lr}")
    for epoch in range(1, epochs + 1):
        train_obj = train_fn(model, loader, optimizer, beta)
        val = evaluate(model, val_loader, beta)
        for key, value in [("epoch", epoch), ("train_objective", train_obj), ("val_recon", val["recon"]),
                           ("val_kl", val["kl"]), ("val_neg_elbo", val["neg_elbo"])]:
            history[key].append(value)
        print(f"  época {epoch:2d} | objetivo (entrenamiento) {train_obj:10.2f} | validación: recon {val['recon']:8.2f} "
              f"· KL {val['kl']:8.2f} · −ELBO {val['neg_elbo']:8.2f} | {time.time() - start:4.0f} s")
    return model, history


@torch.no_grad()
def latent_means(model, x):
    model.eval()
    mu, _ = model.encode(x.to(DEVICE))
    return mu.cpu()


def summarize(model, beta):
    """Métricas de validación de un modelo entrenado (por imagen) + dimensiones latentes activas."""
    stats = evaluate(model, val_loader, beta)
    stats["active_dims"] = int((latent_means(model, x_val).var(dim=0) > 0.01).sum())
    return stats


@torch.no_grad()
def recon_per_image(model, x):
    """BCE sumada sobre los píxeles para cada imagen, decodificando la media μ (sin muestreo)."""
    model.eval()
    x = x.to(DEVICE)
    mu, _ = model.encode(x)
    logits = model.decode(mu)
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1).cpu()


@torch.no_grad()
def reconstruct(model, x):
    model.eval()
    mu, _ = model.encode(x.to(DEVICE))
    return torch.sigmoid(model.decode(mu)).cpu()


@torch.no_grad()
def sample(model, n=12, seed=RUN_SEED):
    model.eval()
    z = torch.randn(n, model.latent_dim, generator=torch.Generator().manual_seed(seed)).to(DEVICE)
    return torch.sigmoid(model.decode(z)).cpu()


def show_images(images, ncols=12, size=1.1, title=None):
    images = torch.as_tensor(images).detach().float().cpu().reshape(-1, 28, 28)
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows + (0.4 if title else 0)))
    for i, ax in enumerate(np.array(axes).reshape(-1)):
        ax.axis("off")
        if i < len(images):
            ax.imshow(images[i], cmap="gray", vmin=0, vmax=1)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_histories(histories):
    """Curvas de validación (reconstrucción y KL por época) de varios modelos."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for label, hist in histories.items():
        axes[0].plot(hist["epoch"], hist["val_recon"], "o-", label=label)
        axes[1].plot(hist["epoch"], hist["val_kl"], "s--", label=label)
    axes[0].set_title("Reconstrucción en validación (más bajo = mejor)")
    axes[1].set_title("KL en validación")
    for ax in axes:
        ax.set_xlabel("época")
        ax.legend()
    plt.tight_layout()
    plt.show()


def print_table(rows):
    """Imprime una tabla de métricas {nombre: stats} (notación científica para valores enormes)."""
    fmt = lambda v: f"{v:10.2f}" if abs(v) < 1e6 else f"{v:10.2e}"
    print(f"{'Modelo':34s} {'recon':>10s} {'KL':>10s} {'−ELBO':>10s} {'objetivo':>10s} {'dims activas':>13s}")
    for name, s in rows.items():
        print(f"{name:34s} {fmt(s['recon'])} {fmt(s['kl'])} {fmt(s['neg_elbo'])} {fmt(s['objective'])} {s['active_dims']:13d}")

# Imágenes de ejemplo de tus prendas (4 de cada una) para comparar reconstrucciones
example_idx = torch.cat([(y_val == c).nonzero().flatten()[:4] for c in CONFIG["classes"]])
x_examples = x_val[example_idx]
show_images(x_examples, title="Ejemplos de validación de tus 3 prendas")

## Tarea 1 — Entrena tu VAE (Ejecución técnica)

1. Ejecuta la celda siguiente: entrena el VAE con **tu** configuración (`CONFIG`) y muestra sus métricas de validación,
   sus reconstrucciones y 12 muestras generadas desde el prior $z \sim \mathcal{N}(0, I)$.
2. Completa la segunda celda: la **pérdida de reconstrucción media por prenda** (el andamiaje ya funciona;
   revísalo y asegúrate de entender cada línea, porque puede salir en la micro-sustentación).
3. Responde en la celda de respuesta.

In [ ]:
modelo_personal, hist_personal = train_vae(CONFIG["latent_dim"], CONFIG["beta"], CONFIG["lr"], label="Tu VAE")
res_personal = summarize(modelo_personal, CONFIG["beta"])
print()
print_table({"Tu VAE": res_personal})

show_images(torch.cat([x_examples, reconstruct(modelo_personal, x_examples)]),
            title="Tu VAE · arriba: originales · abajo: reconstrucciones")
show_images(sample(modelo_personal), title="Tu VAE · 12 muestras desde z ~ N(0, I)")

In [ ]:
# ✏️ TU CÓDIGO AQUÍ: reconstrucción media (BCE por imagen) de cada una de tus 3 prendas.
# Pista: recon_per_image(modelo, x) devuelve un valor por imagen; filtra x_val con y_val == c.
recon_por_prenda = {}
for c in CONFIG["classes"]:
    mask = y_val == c
    recon_por_prenda[GARMENTS[c]] = float(recon_per_image(modelo_personal, x_val[mask]).mean())

for prenda, valor in sorted(recon_por_prenda.items(), key=lambda kv: kv[1]):
    print(f"  {prenda:12s} {valor:8.2f}")
print("Prenda más difícil de reconstruir:", max(recon_por_prenda, key=recon_por_prenda.get))

**Tu respuesta:**

a) Reporta tus métricas de validación: reconstrucción = …, KL = …, −ELBO = …, dimensiones activas = … de ….

b) ¿Cuál de tus 3 prendas se reconstruye peor? Propón una explicación (forma, detalles, variedad dentro de la clase…).

c) Describe en 2–3 líneas tus 12 muestras generadas: ¿se reconocen tus prendas? ¿hay mezclas o imágenes borrosas?

## Tarea 2 — Compara con la línea base (Análisis e interpretación)

La **línea base** usa latente 8 y β = 1 con tu misma tasa de aprendizaje, tus mismas prendas, el mismo número de
épocas y la misma semilla: solo cambian la dimensión latente y β. Ejecuta las celdas y luego interpreta.

Recuerda la diapositiva del β-VAE: compara con **−ELBO = reconstrucción + KL** (no con el objetivo ponderado por β,
que no es comparable entre modelos con distinto β) y mira la reconstrucción y el KL **por separado**.

In [ ]:
modelo_base, hist_base = train_vae(BASELINE["latent_dim"], BASELINE["beta"], BASELINE["lr"], label="Línea base")
res_base = summarize(modelo_base, BASELINE["beta"])
print()
print_table({f"Tu VAE (latente {CONFIG['latent_dim']}, β = {CONFIG['beta']})": res_personal,
             "Línea base (latente 8, β = 1)": res_base})
print(f"\nDiferencia (tu VAE − línea base): recon {res_personal['recon'] - res_base['recon']:+.2f} · "
      f"KL {res_personal['kl'] - res_base['kl']:+.2f} · −ELBO {res_personal['neg_elbo'] - res_base['neg_elbo']:+.2f}")

In [ ]:
plot_histories({"Tu VAE": hist_personal, "Línea base": hist_base})
show_images(torch.cat([x_examples, reconstruct(modelo_personal, x_examples), reconstruct(modelo_base, x_examples)]),
            title="Originales · tu VAE · línea base")
show_images(torch.cat([sample(modelo_personal), sample(modelo_base)]),
            title="Muestras desde el prior · arriba: tu VAE · abajo: línea base")

**Tu respuesta:** (usa **tus** números; entre 8 y 12 líneas)

a) ¿Qué modelo reconstruye mejor y cuál tiene menor KL? Cita las cifras.

b) Explica la diferencia con el papel de β y de la dimensión latente: ¿qué término “ganó” el tira y afloja en cada modelo?
   ¿Qué te dicen las dimensiones activas?

c) Compara las muestras generadas: ¿cuál produce prendas más plausibles o más variadas? ¿Coincide con los números?

d) Según el −ELBO, ¿cuál es mejor modelo generativo de tus datos? ¿Es la misma respuesta que para “mejor reconstrucción”?

## Tarea 3 — Explica y decide: el generador de bocetos de Hilo Norte (Análisis e interpretación)

**Hilo Norte** (empresa ficticia) es un minorista de ropa. Su equipo de diseño quiere una herramienta interna que
genere **bocetos de nuevas prendas** en baja resolución (como Fashion-MNIST) para inspirar colecciones. Requisitos:

1. Generar muchos bocetos **plausibles y variados** a partir de ruido.
2. **Mezclar** dos productos exitosos (interpolación en el espacio latente).
3. **Reentrenar cada mes** con el catálogo nuevo, en una GPU gratuita o pequeña, en minutos.
4. Las imágenes del catálogo son **propias** y no deben salir de la empresa.

Recomienda una configuración y justifícala con tus resultados de las Tareas 1 y 2.

**Tu respuesta:** (entre 10 y 15 líneas)

- **Configuración recomendada** (dimensión latente, β, tasa de aprendizaje, épocas): …
- **Justificación con al menos 2 de tus números** (reconstrucción, KL, −ELBO, dimensiones activas, calidad de las muestras): …
- **¿VAE, GAN o difusión** para este caso? Argumenta con costo, control (interpolación), estabilidad y calidad: …
- **Un riesgo** (técnico, legal o de negocio) y cómo lo mitigarías: …
- **Qué medirías en un piloto** con los diseñadores para decidir si la herramienta sirve: …

## Tarea 4 — Crítica a la IA: un ciclo de entrenamiento con errores (Ejecución técnica + Análisis)

Le pedimos a un asistente de IA una función para entrenar una época del VAE. Respondió con el código de abajo y
añadió: *“Probé el ciclo: la pérdida baja en cada lote, así que el entrenamiento funciona correctamente.”*

El código tiene **3 errores**. Tu trabajo:
1. Ejecuta el código de la IA durante **1 época** y observa la pérdida de entrenamiento y las métricas de validación
   (calculadas con la función `evaluate` correcta).
2. Encuentra los 3 errores y explica, para cada uno, **por qué** es un error y qué síntoma produce.
3. Corrige la función `entrenar_una_epoca_corregida` y vuelve a entrenar con tu configuración.
   **Autoverificación:** si corregiste todo, tus métricas de validación deben coincidir (o casi) con las de la Tarea 1.

In [ ]:
# 🤖 Código generado por un asistente de IA (no lo edites aquí: corrige la copia de más abajo)
def entrenar_una_epoca_ia(model, loader, optimizer, beta=1.0):
    """Entrena una época del VAE y devuelve la pérdida promedio por imagen."""
    model.eval()  # modo determinista para que el entrenamiento sea reproducible
    total, n = 0.0, 0
    for i, (x, _) in enumerate(loader):
        x = x.to(DEVICE)
        logits, mu, logvar = model(x)
        recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.size(0)
        kl = 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        loss = recon + beta * kl
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
        n += x.size(0)
        if i % 40 == 0:
            print(f"    lote {i:3d} | pérdida {loss.item():14.2f} | recon {recon.item():10.2f} | KL {kl.item():14.2f}")
    return total / n

modelo_ia, hist_ia = train_vae(CONFIG["latent_dim"], CONFIG["beta"], CONFIG["lr"], epochs=1,
                               train_fn=entrenar_una_epoca_ia, label="Código de la IA (1 época)")
res_ia = summarize(modelo_ia, CONFIG["beta"])

In [ ]:
# ✏️ TU CÓDIGO AQUÍ: corrige los 3 errores de esta copia del código de la IA.
def entrenar_una_epoca_corregida(model, loader, optimizer, beta=1.0):
    """Entrena una época del VAE y devuelve la pérdida promedio por imagen."""
    model.eval()  # modo determinista para que el entrenamiento sea reproducible
    total, n = 0.0, 0
    for i, (x, _) in enumerate(loader):
        x = x.to(DEVICE)
        logits, mu, logvar = model(x)
        recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.size(0)
        kl = 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        loss = recon + beta * kl
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
        n += x.size(0)
        if i % 40 == 0:
            print(f"    lote {i:3d} | pérdida {loss.item():14.2f} | recon {recon.item():10.2f} | KL {kl.item():14.2f}")
    return total / n

In [ ]:
modelo_corregido, hist_corregido = train_vae(CONFIG["latent_dim"], CONFIG["beta"], CONFIG["lr"],
                                             train_fn=entrenar_una_epoca_corregida, label="Tu versión corregida")
res_corregido = summarize(modelo_corregido, CONFIG["beta"])
print()
print_table({"Código de la IA (1 época)": res_ia, "Tu versión corregida": res_corregido, "Tarea 1 (referencia)": res_personal})
diferencia = abs(res_corregido["neg_elbo"] - res_personal["neg_elbo"])
if res_corregido["kl"] >= 0 and diferencia < 1.0:
    print(f"\n✅ Autoverificación superada: tu −ELBO difiere en {diferencia:.3f} del de la Tarea 1.")
else:
    print(f"\n⚠️ Tu versión corregida aún no coincide con la Tarea 1 (diferencia de −ELBO: {diferencia:.2f}). Revisa los 3 errores.")

**Tu respuesta:**

| # | Línea o instrucción con error | ¿Por qué es un error? ¿Qué síntoma produce? | Corrección |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

**¿Por qué la frase “la pérdida baja en cada lote, así que funciona” era engañosa en este caso?** (3–4 líneas, cita
los números que imprimió la ejecución del código de la IA)

## Resultados (no editar)
Esta celda reúne tus números clave (redondeados a 4 decimales) para la huella de resultados.

In [ ]:
def _round_stats(stats):
    return {k: (int(v) if k == "active_dims" else round(float(v), 4)) for k, v in stats.items()}

RESULTS = {
    "tu_vae": _round_stats(res_personal),
    "linea_base": _round_stats(res_base),
    "recon_por_prenda": {k: round(v, 4) for k, v in recon_por_prenda.items()},
    "codigo_ia_1_epoca": _round_stats(res_ia),
    "version_corregida": _round_stats(res_corregido),
    "epocas": EPOCHS,
}
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))